# Function Vectors Reproduction (nnsight)

Reproduction of Todd et al. (2023) *Function Vectors in Large Language Models* using the **nnsight** library instead of baukit.

**Pipeline:**
1. Collect mean attention head activations over 100 ICL prompts
2. Compute universal function vector from top heads
3. Intervene by adding FV to layer output
4. Evaluate: compare clean vs intervened logits

**Model:** EleutherAI/gpt-j-6b (28 layers, 25 heads, 4096 dim, 164 head_dim)

## 1. Setup & Imports

In [ ]:
import json
import random
import re
import os
from pathlib import Path

import torch
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

from nnsight import LanguageModel

# â”€â”€â”€ GPT-J config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
N_HEADS = 25
N_LAYERS = 28
RESID_DIM = 4096
HEAD_DIM = 164  # RESID_DIM // N_HEADS
PREPEND_BOS = False

# Universal top heads for GPT-J (from Todd et al.)
UNIVERSAL_TOP_INDICES = [
    (15, 5), (9, 14), (12, 10), (8, 1), (11, 0),
    (13, 13), (8, 0), (14, 9), (9, 2), (24, 6),
]

# Edit layer for intervention
EDIT_LAYER = 9

# Number of trials for collecting activations
N_TRIALS = 100
N_ICL_EXAMPLES = 10

# Dataset directory (relative to repo root)
DATASET_DIR = Path("function_vectors/dataset_files/abstractive")

## 2. Utility Functions (ported from Eric's code)

In [ ]:
# â”€â”€â”€ Prompt Utilities â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def load_dataset(dataset_name):
    """Load a JSON dataset from the abstractive dataset files.
    Returns list of [input, target] pairs.
    """
    path = DATASET_DIR / f"{dataset_name}.json"
    with open(path, 'r') as f:
        data = json.load(f)
    return data


def word_pairs_to_prompt_data(word_pairs, n_examples, shuffle=True):
    """Convert word pairs into prompt data dicts."""
    if shuffle:
        random.shuffle(word_pairs)
    prompt_data = []
    for i in range(min(n_examples, len(word_pairs))):
        inp, tgt = word_pairs[i]
        prompt_data.append({
            'input': inp,
            'target': tgt,
            'input_tokens': inp.split(),
            'target_tokens': tgt.split(),
        })
    return prompt_data


def create_prompt(prompt_data, include_target=True, delimiter=' '):
    """Create a prompt string from prompt data."""
    parts = []
    for i, data in enumerate(prompt_data):
        if i > 0:
            parts.append(delimiter)
        parts.append(data['input'])
        if include_target:
            parts.append(delimiter)
            parts.append(data['target'])
    return ''.join(parts)


def get_token_meta_labels(prompt_data, tokenizer, include_target=True, delimiter=' '):
    """Get token-level metadata: which word each token belongs to.
    Returns: (tokens, word_labels) where word_labels[i] = (word_idx, word_type)
    """
    prompt = create_prompt(prompt_data, include_target=include_target, delimiter=delimiter)
    tokens = tokenizer.tokenize(prompt)
    
    word_labels = []
    word_idx = 0
    
    for i, data in enumerate(prompt_data):
        # Input tokens
        input_tokens = tokenizer.tokenize(data['input'])
        for _ in input_tokens:
            word_labels.append((word_idx, 'input'))
        word_idx += 1
        
        # Target tokens
        if include_target:
            target_tokens = tokenizer.tokenize(data['target'])
            for _ in target_tokens:
                word_labels.append((word_idx, 'target'))
            word_idx += 1
    
    return tokens, word_labels


def build_icl_prompt(dataset, n_examples, shuffle=True):
    """Build a complete ICL prompt with token metadata."""
    prompt_data = word_pairs_to_prompt_data(dataset, n_examples, shuffle=shuffle)
    prompt = create_prompt(prompt_data, include_target=True)
    return prompt, prompt_data

In [ ]:
# â”€â”€â”€ Evaluation Utilities â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def decode_to_vocab(logits, tokenizer, top_k=10):
    """Decode logits to top-k vocabulary tokens."""
    probs = torch.softmax(logits.float(), dim=-1)
    top_probs, top_indices = torch.topk(probs, top_k, dim=-1)
    
    results = []
    for idx, prob in zip(top_indices.flatten(), top_probs.flatten()):
        token = tokenizer.decode([idx.item()])
        results.append((token, prob.item()))
    return results


def get_target_token_position(prompt, prompt_data, tokenizer):
    """Find the token position of the last target token in the prompt."""
    tokens = tokenizer.tokenize(prompt)
    return len(tokens) - 1

## 3. Load Model

In [ ]:
# Load GPT-J via nnsight LanguageModel
model = LanguageModel("EleutherAI/gpt-j-6b", device_map="auto", dispatch=True, torch_dtype=torch.float32)
tokenizer = model.tokenizer
device = next(model.parameters()).device
print(f"Model loaded on device: {device}")
print(f"Tokenizer bos_token: {tokenizer.bos_token}, prepend_bos: {PREPEND_BOS}")

## 4. Verify Module Structure

In Eric's baukit code, `attn_hook_names` uses `transformer.h.{L}.attn.out_proj` with `retain_input=True`. This captures the **input to `out_proj`**, which is the concatenated head outputs before the final linear projection â€” shape `(B, S, n_heads * head_dim)`.

In nnsight, the equivalent is `model.transformer.h[L].attn.out_proj.input`.

In [ ]:
# Verify GPT-J attention module structure
print("=== GPT-J Attention Structure ===")
attn = model.transformer.h[0].attn
print(f"Attention module type: {type(attn).__name__}")
print(f"Attention children: {list(attn._modules.keys())}")
print()

# Check out_proj
if hasattr(attn, 'out_proj'):
    print(f"out_proj: {attn.out_proj}")
    print(f"out_proj in_features: {attn.out_proj.in_features}")
    print(f"out_proj out_features: {attn.out_proj.out_features}")
else:
    print("No out_proj found")
    for name, mod in attn.named_modules():
        if 'proj' in name.lower():
            print(f"  {name}: {mod}")

In [ ]:
# Verify what nnsight returns for attn.out_proj.input
input_ids = tokenizer("Hello world", return_tensors='pt').to(device)

with model.trace(input_ids):
    out_proj_input = model.transformer.h[0].attn.out_proj.input.save()
    attn_output = model.transformer.h[0].attn.output.save()

print(f"out_proj_input type: {type(out_proj_input)}")
if isinstance(out_proj_input, tuple):
    for i, t in enumerate(out_proj_input):
        if hasattr(t, 'shape'):
            print(f"  [{i}] shape: {t.shape}")
elif hasattr(out_proj_input, 'shape'):
    print(f"  Shape: {out_proj_input.shape}")

print(f"\nattn_output type: {type(attn_output)}")
if isinstance(attn_output, tuple):
    for i, t in enumerate(attn_output):
        if hasattr(t, 'shape'):
            print(f"  [{i}] shape: {t.shape}")
elif hasattr(attn_output, 'shape'):
    print(f"  Shape: {attn_output.shape}")

## 5. Collect Mean Head Activations

For each of N_TRIALS, we:
1. Sample N_ICL_EXAMPLES pairs from the dataset
2. Build an ICL prompt
3. Trace all 28 layers' `attn.out_proj.input` in forward-pass order
4. Split activations by head via reshape, average multi-token words
5. Accumulate into a mean tensor of shape `(N_LAYERS, N_HEADS, n_words, HEAD_DIM)`

In [ ]:
def collect_mean_head_activations(model, tokenizer, dataset, n_trials=N_TRIALS, 
                                   n_icl_examples=N_ICL_EXAMPLES, device='cuda'):
    """Collect mean attention head activations over n_trials ICL prompts.
    
    For each trial:
    1. Build ICL prompt from shuffled dataset
    2. Trace all 28 layers' attn.out_proj.input in forward-pass order
    3. Split by head: reshape(B, S, n_heads, head_dim)
    4. Average tokens belonging to the same word
    
    Returns:
        mean_activations: tensor of shape (N_LAYERS, N_HEADS, max_words, HEAD_DIM)
    """
    all_trial_results = []
    
    for trial in tqdm(range(n_trials), desc="Collecting activations"):
        # Build prompt
        prompt, prompt_data = build_icl_prompt(dataset, n_icl_examples, shuffle=True)
        
        # Get token metadata for word-level averaging
        tokens, word_labels = get_token_meta_labels(prompt_data, tokenizer, include_target=True)
        n_tokens = len(tokens)
        n_words = len(prompt_data) * 2  # input + target per example
        
        # Tokenize
        inputs = tokenizer(prompt, return_tensors='pt').to(device)
        input_ids = inputs['input_ids']
        
        # Trace: access all 28 layers' attn.out_proj.input in forward-pass order
        with model.trace(input_ids):
            saved_inputs = []
            for layer_idx in range(N_LAYERS):
                saved_inputs.append(
                    model.transformer.h[layer_idx].attn.out_proj.input.save()
                )
        
        # Process saved activations outside the trace
        trial_activations = []
        
        for layer_idx in range(N_LAYERS):
            # saved_inputs[layer_idx] shape: (1, seq_len, resid_dim)
            raw = saved_inputs[layer_idx]
            
            # Split by head: (1, S, n_heads, head_dim)
            by_head = raw.view(1, n_tokens, N_HEADS, HEAD_DIM)
            
            # Average tokens belonging to the same word
            word_activations = []
            for word_idx in range(n_words):
                mask = [i for i, (w, _) in enumerate(word_labels) if w == word_idx]
                if mask:
                    word_avg = by_head[:, mask, :, :].mean(dim=1)
                else:
                    word_avg = torch.zeros(1, N_HEADS, HEAD_DIM, device=device)
                word_activations.append(word_avg)
            
            # Stack: (n_words, n_heads, HEAD_DIM) â†’ transpose to (n_heads, n_words, HEAD_DIM)
            layer_word_activations = torch.cat(word_activations, dim=0)
            layer_word_activations = layer_word_activations.permute(1, 0, 2)
            trial_activations.append(layer_word_activations)
        
        # Stack layers: (N_LAYERS, N_HEADS, n_words, HEAD_DIM)
        trial_tensor = torch.stack(trial_activations)
        all_trial_results.append(trial_tensor)
    
    # Stack all trials and average
    mean_activations = torch.stack(all_trial_results).mean(dim=0)
    return mean_activations

In [ ]:
# Load dataset and collect activations
dataset = load_dataset("country-capital")
print(f"Dataset loaded: {len(dataset)} pairs")
print(f"Sample pair: {dataset[0]}")

# Collect mean head activations (set n_trials=5 for quick test, N_TRIALS=100 for full run)
mean_activations = collect_mean_head_activations(
    model, tokenizer, dataset, 
    n_trials=5,
    n_icl_examples=N_ICL_EXAMPLES,
    device=str(device)
)
print(f"Mean activations shape: {mean_activations.shape}")
# Expected: (28, 25, 20, 164) = (layers, heads, words, head_dim)

## 6. Compute Universal Function Vector

For each universal top head (L, H):
1. Create a zero tensor of shape `(1, resid_dim)`
2. Slot `mean_activations[L, H, T=-1]` (last word's activation) into the head's slice
3. Pass through `out_proj` to get the contribution to the residual stream
4. Sum all contributions to get the FV vector

In [ ]:
def compute_universal_function_vector(mean_activations, model, 
                                       top_indices=UNIVERSAL_TOP_INDICES,
                                       device='cuda'):
    """Compute the universal function vector from mean head activations.
    
    Args:
        mean_activations: (N_LAYERS, N_HEADS, n_words, HEAD_DIM)
    
    Returns:
        fv_vector: (1, RESID_DIM) tensor
    """
    fv_vector = torch.zeros(1, RESID_DIM, device=device)
    
    for layer_idx, head_idx in top_indices:
        # Get the last word's activation for this head
        head_activation = mean_activations[layer_idx, head_idx, -1, :]  # (HEAD_DIM,)
        
        # Create zero resid_dim tensor and slot in head's slice
        head_slot = torch.zeros(RESID_DIM, device=device)
        start = head_idx * HEAD_DIM
        end = start + HEAD_DIM
        head_slot[start:end] = head_activation
        
        # Pass through out_proj
        out_proj = model.transformer.h[layer_idx].attn.out_proj
        d_out = out_proj(head_slot.unsqueeze(0))  # (1, RESID_DIM)
        
        fv_vector = fv_vector + d_out
    
    return fv_vector


# Compute FV
fv_vector = compute_universal_function_vector(mean_activations, model, device=str(device))
print(f"FV vector shape: {fv_vector.shape}")
print(f"FV vector norm: {fv_vector.norm().item():.4f}")

## 7. Intervention Experiment

Test whether adding the FV to the model's computation changes its behavior.

1. Run a clean forward pass to get baseline logits
2. Run an intervened forward pass (add FV at layer 9)
3. Compare the top-k predictions

In [ ]:
def run_clean_forward(prompt, model, tokenizer, device='cuda'):
    """Run a clean forward pass and return last-token logits."""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad(), model.trace(inputs['input_ids']):
        output = model.output.save()
    
    if hasattr(output, 'logits'):
        logits = output.logits[0, -1, :]
    else:
        logits = output[0, -1, :]
    return logits


def run_intervened_forward(prompt, model, tokenizer, fv_vector, 
                           edit_layer=EDIT_LAYER, device='cuda'):
    """Run intervened forward pass: add FV to layer output.
    
    In nnsight, model.transformer.h[L].output is a tuple (hidden_states, ...).
    We modify hidden_states: hidden[:, -1, :] += fv_vector
    """
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    
    with model.trace(inputs['input_ids']):
        # Get the output of the edit layer's transformer block
        hidden = model.transformer.h[edit_layer].output[0]
        
        # Add FV to the last token position (assignment, not in-place)
        hidden[:, -1, :] = hidden[:, -1, :] + fv_vector
        
        # Get output logits
        output = model.output.save()
    
    if hasattr(output, 'logits'):
        logits = output.logits[0, -1, :]
    else:
        logits = output[0, -1, :]
    return logits

In [ ]:
# Test intervention on a shuffled-label ICL prompt
random.seed(42)
dataset = load_dataset("country-capital")

# Build a shuffled prompt
prompt_data = word_pairs_to_prompt_data(dataset, N_ICL_EXAMPLES, shuffle=True)
prompt = create_prompt(prompt_data, include_target=True)

print(f"Prompt: {prompt[:200]}...")
print()

# Clean forward pass
clean_logits = run_clean_forward(prompt, model, tokenizer, device=str(device))
clean_top = decode_to_vocab(clean_logits, tokenizer, top_k=10)

print("=== Clean top-10 predictions ===")
for token, prob in clean_top:
    print(f"  {token:20s} {prob:.4f}")

# Intervened forward pass
intervened_logits = run_intervened_forward(prompt, model, tokenizer, fv_vector, 
                                            edit_layer=EDIT_LAYER, device=str(device))
intervened_top = decode_to_vocab(intervened_logits, tokenizer, top_k=10)

print("\n=== Intervened top-10 predictions ===")
for token, prob in intervened_top:
    print(f"  {token:20s} {prob:.4f}")

## 8. Evaluation

Evaluate the FV intervention across different prompt types:
1. **ICL (shuffled labels)**: Does FV make the model follow the pattern even with wrong labels?
2. **Zero-shot**: Does FV induce the task behavior without examples?

In [ ]:
def evaluate_icl_intervention(model, tokenizer, fv_vector, dataset_name="country-capital",
                               n_trials=10, n_icl_examples=10, edit_layer=EDIT_LAYER, 
                               device='cuda'):
    """Evaluate FV intervention on ICL prompts with shuffled labels."""
    dataset = load_dataset(dataset_name)
    
    clean_correct_probs = []
    intervened_correct_probs = []
    
    for trial in tqdm(range(n_trials), desc=f"Evaluating {dataset_name}"):
        # Build shuffled prompt
        prompt_data = word_pairs_to_prompt_data(dataset, n_icl_examples, shuffle=True)
        prompt = create_prompt(prompt_data, include_target=True)
        
        # Get the next input (the query without target)
        query_data = word_pairs_to_prompt_data(dataset, 1, shuffle=False)
        query = prompt + " " + query_data[0]['input']
        
        # The correct answer (following the pattern)
        correct_target = query_data[0]['target']
        
        # Clean forward pass
        clean_logits = run_clean_forward(query, model, tokenizer, device=device)
        
        # Intervened forward pass
        intervened_logits = run_intervened_forward(query, model, tokenizer, fv_vector,
                                                    edit_layer=edit_layer, device=device)
        
        # Get probability of correct target
        target_tokens = tokenizer.tokenize(correct_target)
        target_ids = tokenizer.convert_tokens_to_ids(target_tokens)
        if isinstance(target_ids, int):
            target_ids = [target_ids]
        
        clean_probs = torch.softmax(clean_logits.float(), dim=-1)
        intervened_probs = torch.softmax(intervened_logits.float(), dim=-1)
        
        clean_correct_probs.append(clean_probs[target_ids[0]].item())
        intervened_correct_probs.append(intervened_probs[target_ids[0]].item())
    
    return {
        'clean_mean': np.mean(clean_correct_probs),
        'clean_std': np.std(clean_correct_probs),
        'intervened_mean': np.mean(intervened_correct_probs),
        'intervened_std': np.std(intervened_correct_probs),
    }


results = evaluate_icl_intervention(model, tokenizer, fv_vector, 
                                     dataset_name="country-capital",
                                     n_trials=10, device=str(device))

print(f"\n=== ICL Evaluation (country-capital) ===")
print(f"Clean:       {results['clean_mean']:.4f} Â± {results['clean_std']:.4f}")
print(f"Intervened:  {results['intervened_mean']:.4f} Â± {results['intervened_std']:.4f}")
print(f"Delta:       {results['intervened_mean'] - results['clean_mean']:.4f}")

In [ ]:
# Zero-shot evaluation
def evaluate_zeroshot_intervention(model, tokenizer, fv_vector, dataset_name="country-capital",
                                    n_trials=10, edit_layer=EDIT_LAYER, device='cuda'):
    """Evaluate FV intervention on zero-shot prompts (no ICL examples)."""
    dataset = load_dataset(dataset_name)
    
    clean_probs = []
    intervened_probs = []
    
    for trial in tqdm(range(n_trials), desc=f"Zero-shot {dataset_name}"):
        inp, tgt = random.choice(dataset)
        query = inp
        
        clean_logits = run_clean_forward(query, model, tokenizer, device=device)
        intervened_logits = run_intervened_forward(query, model, tokenizer, fv_vector,
                                                    edit_layer=edit_layer, device=device)
        
        target_tokens = tokenizer.tokenize(tgt)
        target_ids = tokenizer.convert_tokens_to_ids(target_tokens)
        if isinstance(target_ids, int):
            target_ids = [target_ids]
        
        clean_probs.append(torch.softmax(clean_logits.float(), dim=-1)[target_ids[0]].item())
        intervened_probs.append(torch.softmax(intervened_logits.float(), dim=-1)[target_ids[0]].item())
    
    return {
        'clean_mean': np.mean(clean_probs),
        'clean_std': np.std(clean_probs),
        'intervened_mean': np.mean(intervened_probs),
        'intervened_std': np.std(intervened_probs),
    }


zeroshot_results = evaluate_zeroshot_intervention(model, tokenizer, fv_vector,
                                                   dataset_name="country-capital",
                                                   n_trials=10, device=str(device))

print(f"\n=== Zero-shot Evaluation (country-capital) ===")
print(f"Clean:       {zeroshot_results['clean_mean']:.4f} Â± {zeroshot_results['clean_std']:.4f}")
print(f"Intervened:  {zeroshot_results['intervened_mean']:.4f} Â± {zeroshot_results['intervened_std']:.4f}")
print(f"Delta:       {zeroshot_results['intervened_mean'] - zeroshot_results['clean_mean']:.4f}")

## 9. Visualization

In [ ]:
# Visualize mean head activations for the universal top heads
fig, axes = plt.subplots(2, 5, figsize=(20, 6))
axes = axes.flatten()

for i, (layer_idx, head_idx) in enumerate(UNIVERSAL_TOP_INDICES):
    activations = mean_activations[layer_idx, head_idx, :, :].cpu().numpy()
    norms = np.linalg.norm(activations, axis=1)
    axes[i].plot(norms, marker='o', markersize=3)
    axes[i].set_title(f"Layer {layer_idx}, Head {head_idx}")
    axes[i].set_xlabel("Word position")
    axes[i].set_ylabel("Activation norm")

plt.suptitle("Universal Top Heads: Activation Norms Across Word Positions", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize clean vs intervened logit distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

clean_probs = torch.softmax(clean_logits.float(), dim=-1).cpu().numpy()
top_clean_idx = np.argsort(clean_probs)[-20:][::-1]
top_clean_tokens = [tokenizer.decode([idx]) for idx in top_clean_idx]
top_clean_probs = clean_probs[top_clean_idx]

axes[0].barh(range(len(top_clean_tokens)), top_clean_probs)
axes[0].set_yticks(range(len(top_clean_tokens)))
axes[0].set_yticklabels(top_clean_tokens)
axes[0].set_title("Clean top-20 predictions")
axes[0].invert_yaxis()

intervened_probs = torch.softmax(intervened_logits.float(), dim=-1).cpu().numpy()
top_int_idx = np.argsort(intervened_probs)[-20:][::-1]
top_int_tokens = [tokenizer.decode([idx]) for idx in top_int_idx]
top_int_probs = intervened_probs[top_int_idx]

axes[1].barh(range(len(top_int_tokens)), top_int_probs)
axes[1].set_yticks(range(len(top_int_tokens)))
axes[1].set_yticklabels(top_int_tokens)
axes[1].set_title("Intervened top-20 predictions")
axes[1].invert_yaxis()

plt.suptitle("Clean vs Intervened: Top-20 Token Predictions", fontsize=14)
plt.tight_layout()
plt.show()

## 10. Natural Text Intervention (Optional)

Test FV intervention on natural text generation using `model.generate()`.

In [ ]:
def fv_intervention_natural_text(model, tokenizer, fv_vector, prompt,
                                  edit_layer=EDIT_LAYER, max_new_tokens=50,
                                  device='cuda'):
    """Generate text with FV intervention using model.generate().
    
    Uses model.generate() as the context manager (not model.trace()).
    The intervention is applied at each generation step.
    """
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    
    with model.generate(inputs['input_ids'], max_new_tokens=max_new_tokens):
        # Add FV to the edit layer's output at each generation step
        hidden = model.transformer.h[edit_layer].output[0]
        hidden[:, -1, :] = hidden[:, -1, :] + fv_vector
        
        # Get the generated output
        output = model.generator.output.save()
    
    return tokenizer.decode(output[0], skip_special_tokens=True)


# Test natural text generation
natural_prompt = "The capital of France is"
print(f"Prompt: {natural_prompt}")
print()

# Clean generation
clean_gen = model.generate(natural_prompt, max_new_tokens=50)
print(f"Clean generation: {clean_gen}")

# Intervened generation
intervened_gen = fv_intervention_natural_text(model, tokenizer, fv_vector, natural_prompt,
                                               edit_layer=EDIT_LAYER, max_new_tokens=50,
                                               device=str(device))
print(f"Intervened generation: {intervened_gen}")